# Build cycle_summary (test: 2 files)

Exports all `summary` columns for each cycle. When this looks correct, change `NUM_FILES` to `None` and run a full export to `cycle_summary.csv`.

In [ ]:
import os
import sys
import time
os.chdir('..')
import json
import pandas as pd
import glob

sys.path.insert(0, 'scripts')
from dedupe_policy import is_kept_file

NUM_FILES = None  # set to 2 for a quick test; None = all files
OUTPUT_PATH = 'data/processed/cycle_summary.csv'

def fmt_seconds(seconds):
    """Format seconds as 45.2s or 3m 12s."""
    if seconds < 60:
        return f"{seconds:.1f}s"
    minutes, secs = divmod(int(seconds), 60)
    return f"{minutes}m {secs}s"

os.makedirs('data/processed', exist_ok=True)

files = sorted(glob.glob('data/raw/FastCharge*.json'))
if NUM_FILES is not None:
    files = files[:NUM_FILES]
print(f"Processing {len(files)} file(s)")
print(f"Output: {OUTPUT_PATH}")

In [ ]:
parts = []
loop_start = time.perf_counter()

for i, file in enumerate(files):
    file_id = os.path.basename(file)
    if not is_kept_file(file_id):
        continue

    file_start = time.perf_counter()

    with open(file) as f:
        data = json.load(f)

    summary = pd.DataFrame(data['summary'])
    summary['file_id'] = file_id
    summary['cell_id'] = data['barcode']
    parts.append(summary)

    file_elapsed = time.perf_counter() - file_start
    total_elapsed = time.perf_counter() - loop_start
    done = i + 1
    avg_per_file = total_elapsed / done
    remaining = len(files) - done
    eta = avg_per_file * remaining

    print(
        f"[{done}/{len(files)}] {os.path.basename(file)} | "
        f"this file: {fmt_seconds(file_elapsed)} | "
        f"running total: {fmt_seconds(total_elapsed)} | "
        f"ETA: {fmt_seconds(eta) if remaining else '0s'}"
    )

process_elapsed = time.perf_counter() - loop_start

concat_start = time.perf_counter()
cycle_summary = pd.concat(parts, ignore_index=True)
cols = ['file_id', 'cell_id'] + [c for c in cycle_summary.columns if c not in ('file_id', 'cell_id')]
cycle_summary = cycle_summary[cols]
concat_elapsed = time.perf_counter() - concat_start

print()
print(f"Load + parse all files: {fmt_seconds(process_elapsed)}")
print(f"Concat DataFrame:       {fmt_seconds(concat_elapsed)}")
print(f"Rows: {len(cycle_summary):,}")
print(f"Columns ({len(cycle_summary.columns)}):", list(cycle_summary.columns))

In [ ]:
print("Rows per cell:")
print(cycle_summary.groupby('cell_id').size())

cycle_summary.head(10)

In [ ]:
save_start = time.perf_counter()
cycle_summary.to_csv(OUTPUT_PATH, index=False)
save_elapsed = time.perf_counter() - save_start

print(f"Saved to {OUTPUT_PATH} ({fmt_seconds(save_elapsed)})")
print()
print("--- Timing summary ---")
print(f"Load + parse:  {fmt_seconds(process_elapsed)}")
print(f"Concat:        {fmt_seconds(concat_elapsed)}")
print(f"Write CSV:     {fmt_seconds(save_elapsed)}")
print(f"TOTAL:         {fmt_seconds(process_elapsed + concat_elapsed + save_elapsed)}")